In [1]:
from google.colab import drive
drive.mount("/content/drive")

ValueError: mount failed

In [ ]:
import pandas as pd
import re
import json
import hashlib
import platform
import unicodedata
import scipy.sparse
import joblib
from pathlib import Path
from collections import Counter
from datetime import datetime
from zoneinfo import ZoneInfo
from itertools import combinations
from sklearn.feature_extraction.text import TfidfVectorizer

PROJECT = Path('/content/drive/MyDrive/text_mining_project_01')
ORIGINAL = PROJECT / 'original_data'
CLEAN = PROJECT / 'clean_data'

# ==========================================
# CONFIGURATION: REUSE OR RECALCULATE?
# ==========================================
# Set to None to process raw data and create a new folder.
# Set to 'latest' to load the most recent processing run.
# Set to a specific folder name (e.g., 'clean-data_14-09...') to load that exact run.
LOAD_EXISTING_VERSION = "clean-data_14-09-2026_22-03-20_087693_UTC-0300"
# ==========================================

ARQUIVOS = ['train.csv', 'valid.csv', 'test.csv', 'sample_submission.csv']
ALVOS = ['formal_register', 'thematic_coherence', 'narrative_rhetorical_structure', 'cohesion']
TAMANHOS_ESPERADOS = {'train': 740, 'valid': 125, 'test': 370}

if not ORIGINAL.is_dir():
    raise FileNotFoundError(f'Folder not found. Check the name and location in Drive: {ORIGINAL}')

CLEAN.mkdir(exist_ok=True)

if LOAD_EXISTING_VERSION:
    if LOAD_EXISTING_VERSION == 'latest':
        past_runs = sorted([d for d in CLEAN.iterdir() if d.is_dir()])
        if not past_runs:
            raise FileNotFoundError("No previous runs found in clean_data/")
        SAIDA = past_runs[-1]
    else:
        SAIDA = CLEAN / LOAD_EXISTING_VERSION
        if not SAIDA.exists():
             raise FileNotFoundError(f"Version {LOAD_EXISTING_VERSION} not found!")
    print(f"--> TARGET DIRECTORY SET TO LOAD EXISTING DATA: {SAIDA.name}\n")
else:
    now = datetime.now(ZoneInfo("America/Recife"))
    VERSAO = now.strftime("clean-data_%d-%m-%Y_%H-%M-%S_%f_UTC%z")
    SAIDA = CLEAN / VERSAO
    print(f"--> TARGET DIRECTORY SET TO CREATE NEW DATA: {SAIDA.name}\n")

print('Files found in original data:')
for arquivo in sorted(ORIGINAL.iterdir()):
    print(f" - {arquivo.name}")

faltantes = [nome for nome in ARQUIVOS if not (ORIGINAL / nome).is_file()]
if faltantes:
    raise FileNotFoundError(f'Missing files in {ORIGINAL}: {faltantes}')

# Capture original hashes to ensure data integrity
hashes_originais = {
    nome: hashlib.sha256((ORIGINAL / nome).read_bytes()).hexdigest()
    for nome in ARQUIVOS
}

brutos = {
    nome.replace('.csv', ''): pd.read_csv(ORIGINAL / nome, encoding='utf-8-sig', dtype=str, keep_default_na=False)
    for nome in ['train.csv', 'valid.csv', 'test.csv']
}

print('\nRaw data loaded into memory:')
for nome, df in brutos.items():
    print(f"{nome:<6} | Shape: {df.shape} | Columns: {df.columns.tolist()}")

--> TARGET DIRECTORY SET TO LOAD EXISTING DATA: clean-data_14-09-2026_22-03-20_087693_UTC-0300

Files found in original data:
 - sample_submission.csv
 - test.csv
 - train.csv
 - valid.csv

Raw data loaded into memory:
train  | Shape: (740, 7) | Columns: ['id', 'essay', 'prompt', 'formal_register', 'thematic_coherence', 'narrative_rhetorical_structure', 'cohesion']
valid  | Shape: (125, 7) | Columns: ['id', 'essay', 'prompt', 'formal_register', 'thematic_coherence', 'narrative_rhetorical_structure', 'cohesion']
test   | Shape: (370, 3) | Columns: ['id', 'essay', 'prompt']


### Data Diagnostics (Step 1)
Verification of structural integrity: expected columns, expected row counts, empty values, duplicate IDs, target score validation, and cross-set leak detection.

In [ ]:
dados = {nome: df.copy(deep=True) for nome, df in brutos.items()}
resumo = []
problemas = []

for nome, df in dados.items():
    colunas_esperadas = ["id", "essay", "prompt"]
    if nome in ["train", "valid"]:
        colunas_esperadas += ALVOS

    faltam = set(colunas_esperadas) - set(df.columns)
    extras = set(df.columns) - set(colunas_esperadas)

    if faltam or extras:
        problemas.append(f"{nome}: missing columns={sorted(faltam)}, extra columns={sorted(extras)}")

    if len(df) != TAMANHOS_ESPERADOS[nome]:
        problemas.append(f"{nome}: expected {TAMANHOS_ESPERADOS[nome]} rows, found {len(df)}.")

    if faltam:
        continue

    ids_vazios = df["id"].str.strip().eq("")
    ids_duplicados = df["id"].duplicated(keep=False)
    textos_vazios = df["essay"].str.strip().eq("")
    temas_vazios = df["prompt"].str.strip().eq("")

    if ids_vazios.any(): problemas.append(f"{nome}: contains empty IDs.")
    if ids_duplicados.any(): problemas.append(f"{nome}: contains duplicate IDs.")

    # Validate scores before converting to numeric
    if nome in ["train", "valid"]:
        for alvo in ALVOS:
            notas = pd.to_numeric(df[alvo], errors="coerce")
            invalidas = ~notas.isin([1, 2, 3, 4, 5])
            if invalidas.any():
                problemas.append(f"{nome}: {int(invalidas.sum())} invalid scores in column '{alvo}'.")
            else:
                df[alvo] = notas.astype("int64")

    resumo.append({
        "dataset": nome,
        "rows": len(df),
        "columns": len(df.columns),
        "empty_ids": int(ids_vazios.sum()),
        "duplicate_ids": int(ids_duplicados.sum()),
        "empty_essays": int(textos_vazios.sum()),
        "empty_prompts": int(temas_vazios.sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_essays": int(df["essay"].duplicated().sum()),
    })

diagnostico = pd.DataFrame(resumo)
display(diagnostico)

if problemas:
    raise ValueError("Please review the issues found:\n- " + "\n- ".join(problemas))

print("Structural validation complete. No rows were removed.")

# Check for ID leakage across sets (Train/Valid/Test)
for conjunto_a, conjunto_b in combinations(dados.keys(), 2):
    ids_comuns = set(dados[conjunto_a]["id"]) & set(dados[conjunto_b]["id"])
    if ids_comuns:
        raise ValueError(f"Shared IDs found between {conjunto_a} and {conjunto_b}: {sorted(ids_comuns)}")

print("No IDs are shared across datasets (No data leakage).")

,dataset,rows,columns,empty_ids,duplicate_ids,empty_essays,empty_prompts,duplicate_rows,duplicate_essays
0,train,740,7,0,0,0,0,0,0
1,valid,125,7,0,0,0,0,0,0
2,test,370,3,0,0,0,0,0,0


Structural validation complete. No rows were removed.
No IDs are shared across datasets (No data leakage).


### Conservative Preprocessing (Step 2)
Identification and treatment of text markers, text cleaning without losing punctuation/accents, and creation of auxiliary features.

In [ ]:
if LOAD_EXISTING_VERSION:
    print(f"Loading previously cleaned datasets from: {SAIDA.name} ...")
    limpos = {
        nome: pd.read_csv(SAIDA / f"{nome}_limpo.csv", encoding='utf-8', keep_default_na=False)
        for nome in ['train', 'valid', 'test']
    }
    print("Successfully loaded previous datasets into memory.")

else:
    MARCADORES = {
        "paragrafo": ["[P]", "[ P]", "[P}", "[p]", "{p}"],
        "titulo": ["[T]", "[t]", "{t}"],
        "rasura": ["[R]", "[X]", "[X~]", r"[X\~]", "[r]", "[x]", "{x}"],
        "simbolo": ["[S]", "[s]"],
        "desconhecido": ["[?]", "{?}", "[?}", "{?]"],
        "fora_da_linha": ["[LC]", "[LT]", "[lt]"],
    }

    TOKEN_GRUPO = {token: grupo for grupo, tokens in MARCADORES.items() for token in tokens}

    PADROES = {
        grupo: re.compile("|".join(re.escape(token) for token in sorted(tokens, key=len, reverse=True)))
        for grupo, tokens in MARCADORES.items()
    }

    # Search for possible short tags enclosed in brackets, braces, or angle brackets
    CANDIDATOS = re.compile(r"[\[{][^\[\]{}\n]{0,30}[\]}]|<[^<>\n]{1,30}>")

    def limpar_texto(texto):
        texto = unicodedata.normalize("NFC", texto)
        texto = texto.replace("\r\n", "\n").replace("\r", "\n")

        for grupo, padrao in PADROES.items():
            substituto = "\n" if grupo == "paragrafo" else " "
            texto = padrao.sub(substituto, texto)

        texto = re.sub(r"[^\S\n]+", " ", texto)
        texto = re.sub(r" *\n *", "\n", texto)
        return texto.strip()

    def preparar_dataset(df):
        resultado = df.copy(deep=True)

        for grupo, padrao in PADROES.items():
            resultado[f"n_marcadores_{grupo}"] = df["essay"].map(lambda t: len(padrao.findall(t)))

        resultado["n_marcadores_nao_documentados"] = df["essay"].map(
            lambda t: sum(token not in TOKEN_GRUPO for token in CANDIDATOS.findall(t))
        )

        resultado["essay_clean"] = df["essay"].map(limpar_texto)
        resultado["texto_limpo_vazio"] = resultado["essay_clean"].eq("").astype(int)

        # Additional metadata features
        resultado["n_caracteres"] = resultado["essay_clean"].str.len()
        resultado["n_palavras"] = resultado["essay_clean"].map(lambda t: len(re.findall(r"\b\w+\b", t)))

        return resultado

    limpos = {nome: preparar_dataset(df) for nome, df in dados.items()}
    print("Clean versions created in memory.")

Loading previously cleaned datasets from: clean-data_14-09-2026_22-03-20_087693_UTC-0300 ...
Successfully loaded previous datasets into memory.


In [ ]:
if LOAD_EXISTING_VERSION:
    print(f"Skipping saving procedure. Utilizing existing datasets from {SAIDA.name}")
else:
    RELATORIOS = SAIDA / "relatorios"
    RELATORIOS.mkdir(parents=True, exist_ok=False)

    # Save and verify each CSV
    for nome, df in limpos.items():
        destino = SAIDA / f"{nome}_limpo.csv"
        df.to_csv(destino, index=False, encoding="utf-8")
        print(f"Saved CSV: {destino.name}")

    # Save execution manifesto
    manifesto = {
        "version": VERSAO,
        "python_version": platform.python_version(),
        "pandas_version": pd.__version__,
        "original_hashes": hashes_originais,
        "exported_rows": {nome: len(df) for nome, df in limpos.items()},
        "documented_markers": MARCADORES,
        "undocumented_markers": "preserved",
        "spell_check": False,
        "data_augmentation": False,
    }
    (RELATORIOS / "manifesto.json").write_text(json.dumps(manifesto, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Export complete. Files saved to: {SAIDA}")

Skipping saving procedure. Utilizing existing datasets from clean-data_14-09-2026_22-03-20_087693_UTC-0300


### TF-IDF Vectorization & Saving (Step 2)
Extraction of numerical text features, strictly fitting on the training set to prevent data leakage, and saving the matrices to drive.

In [ ]:
# Check if the TF-IDF matrices already exist in the target directory
caminho_tfidf = SAIDA / "X_train_tfidf.npz"

if caminho_tfidf.exists():
    print(f"Loading existing TF-IDF matrices and vectorizer from: {SAIDA.name} ...")
    X_train = scipy.sparse.load_npz(SAIDA / "X_train_tfidf.npz")
    X_valid = scipy.sparse.load_npz(SAIDA / "X_valid_tfidf.npz")
    X_test  = scipy.sparse.load_npz(SAIDA / "X_test_tfidf.npz")
    vetorizador = joblib.load(SAIDA / "tfidf_vectorizer.joblib")

    print("Successfully loaded TF-IDF assets.")
    print(f"Loaded vocabulary contains {X_train.shape[1]} features.")
    print(f"Train Matrix Shape:      {X_train.shape}")
    print(f"Validation Matrix Shape: {X_valid.shape}")
    print(f"Test Matrix Shape:       {X_test.shape}")

else:
    print(f"TF-IDF files not found in {SAIDA.name}. Generating them now...")
    vetorizador = TfidfVectorizer(
        lowercase=False,
        strip_accents=None,
        stop_words=None,
        ngram_range=(1, 2),
        min_df=2,
    )

    # IMPORTANT: fit_transform learns vocabulary and weights ONLY on the training set
    X_train = vetorizador.fit_transform(limpos["train"]["essay_clean"])

    # transform applies the learned vocabulary to the valid and test sets
    X_valid = vetorizador.transform(limpos["valid"]["essay_clean"])
    X_test = vetorizador.transform(limpos["test"]["essay_clean"])

    print(f"Vocabulary learned on the training set contains {X_train.shape[1]} features.")
    print(f"Train Matrix Shape:      {X_train.shape}")
    print(f"Validation Matrix Shape: {X_valid.shape}")
    print(f"Test Matrix Shape:       {X_test.shape}")

    # Save TF-IDF sparse matrices to the same output folder
    scipy.sparse.save_npz(SAIDA / "X_train_tfidf.npz", X_train)
    scipy.sparse.save_npz(SAIDA / "X_valid_tfidf.npz", X_valid)
    scipy.sparse.save_npz(SAIDA / "X_test_tfidf.npz", X_test)

    # Save the fitted vectorizer so we can use the exact same vocabulary later
    joblib.dump(vetorizador, SAIDA / "tfidf_vectorizer.joblib")

    print(f"\nTF-IDF matrices (.npz) and vectorizer (.joblib) successfully saved to:")
    print(SAIDA)

TF-IDF files not found in clean-data_14-09-2026_22-03-20_087693_UTC-0300. Generating them now...
Vocabulary learned on the training set contains 16989 features.
Train Matrix Shape:      (740, 16989)
Validation Matrix Shape: (125, 16989)
Test Matrix Shape:       (370, 16989)

TF-IDF matrices (.npz) and vectorizer (.joblib) successfully saved to:
/content/drive/MyDrive/text_mining_project_01/clean_data/clean-data_14-09-2026_22-03-20_087693_UTC-0300


## Data Dictionary for New Columns
As requested, here is the explanation for the generated columns:

* **`n_marcadores_paragrafo`**: Count of paragraph tags (e.g., `[P]`) present in the original text.
* **`n_marcadores_titulo`**: Count of title tags (e.g., `[T]`).
* **`n_marcadores_rasura`**: Count of erasure/correction tags (e.g., `[R]`, `[X]`).
* **`n_marcadores_simbolo`**: Count of symbol tags (e.g., `[S]`).
* **`n_marcadores_desconhecido`**: Count of illegible text tags (e.g., `[?]`).
* **`n_marcadores_fora_da_linha`**: Count of margin spill tags (e.g., `[LC]`).
* **`n_marcadores_nao_documentados`**: Dynamic count of unexpected or undocumented bracketed tags found in the text.
* **`essay_clean`**: The processed essay text where markers were removed or replaced (e.g., paragraph markers converted to line breaks), while accents, spelling errors, and valid punctuation are preserved.
* **`texto_limpo_vazio`**: Binary indicator (1 if the text became completely empty after cleaning, else 0).
* **`n_caracteres`**: Total string length of `essay_clean`.
* **`n_palavras`**: Word count estimation using regex word boundaries (`\b\w+\b`).